In [ ]:
from pathlib import Path
import time

import chipwhisperer as cw


Q = 127

INPUT_BYTES = 1
OUTPUT_BYTES = 1

SUBROUTINE_FQ_INV = 0x03
BAUDRATE = 230400

FIRMWARE_DIRECTORY = Path.cwd()

HEX_BINARY = (
    FIRMWARE_DIRECTORY
    / "less_fq_inv-PINATA.hex"
)

print("Input bytes :", INPUT_BYTES)
print("Output bytes:", OUTPUT_BYTES)
print("HEX file    :", HEX_BINARY)

In [ ]:
def serialize_fq_inv(value: int) -> bytes:
    """Serialize one reduced LESS field element."""

    if not isinstance(value, int):
        raise TypeError(
            "Field element must be an integer"
        )

    if value < 0 or value >= Q:
        raise ValueError(
            f"Field element must be in [0, {Q - 1}]"
        )

    return bytes([value])


def fq_inv_python(value: int) -> int:
    """Independent Python calculation."""

    serialize_fq_inv(value)

    if value == 0:
        return 0

    return pow(value, -1, Q)

In [ ]:
def connect_and_flash():
    """Connect, configure the clock and flash the firmware."""

    if not HEX_BINARY.is_file():
        raise FileNotFoundError(
            f"Firmware image not found: {HEX_BINARY}"
        )

    scope = cw.scope()
    scope.default_setup()

    scope.io.clkout = 15e6

    print("Clock output:", scope.io.clkout)

    time.sleep(0.5)

    cw.program_target(
        scope,
        cw.programmers.STM32FProgrammer,
        str(HEX_BINARY),
        baud=BAUDRATE,
    )

    target = cw.target(
        scope,
        cw.targets.SimpleSerial2,
    )

    target.baud = BAUDRATE
    target.flush()

    return scope, target


def upload_fq_inv_input(
    target,
    payload: bytes,
    verify_readback: bool = True,
) -> None:
    """Upload the one-byte fq_inv input."""

    if len(payload) != INPUT_BYTES:
        raise ValueError(
            f"Expected {INPUT_BYTES} input byte, "
            f"got {len(payload)}"
        )

    subcommand = (
        0x80
        if verify_readback
        else 0x00
    )

    target.send_cmd(
        "s",
        subcommand,
        bytearray(payload),
    )

    if verify_readback:
        readback = target.simpleserial_read(
            "r",
            INPUT_BYTES,
            timeout=1000,
            ack=True,
        )

        if readback is None:
            raise TimeoutError(
                "No fq_inv input readback received"
            )

        if bytes(readback) != payload:
            raise RuntimeError(
                "fq_inv input readback mismatch"
            )

    else:
        acknowledgement = (
            target.simpleserial_wait_ack(
                timeout=1000
            )
        )

        if acknowledgement is None:
            raise TimeoutError(
                "No acknowledgement for fq_inv input"
            )


def fq_inv_board(
    target,
    value: int,
) -> int:
    """Execute fq_inv on the Pinata."""

    payload = serialize_fq_inv(value)

    upload_fq_inv_input(
        target,
        payload,
        verify_readback=True,
    )

    target.send_cmd(
        "b",
        SUBROUTINE_FQ_INV,
        bytearray(),
    )

    response = target.simpleserial_read(
        "r",
        OUTPUT_BYTES,
        timeout=1000,
        ack=True,
    )

    if response is None:
        raise TimeoutError(
            "No result received from fq_inv"
        )

    if len(response) != OUTPUT_BYTES:
        raise RuntimeError(
            f"Expected {OUTPUT_BYTES} result byte, "
            f"received {len(response)}"
        )

    return int(response[0])

In [ ]:
scope = None
target = None

try:
    scope, target = connect_and_flash()

    for value in range(Q):
        expected = fq_inv_python(value)
        actual = fq_inv_board(
            target,
            value,
        )

        if actual != expected:
            raise AssertionError(
                f"fq_inv mismatch for {value}: "
                f"expected {expected}, got {actual}"
            )

    print(
        f"OK: all {Q} reduced field elements match."
    )

finally:
    if target is not None:
        target.dis()

    if scope is not None:
        scope.dis()